# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('AZURE_OPENAI_API_KEY')
azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_api_version = os.getenv('AZURE_OPENAI_API_VERSION', '2024-08-01-preview')
azure_deployment_name = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')

if not api_key:
    print("No API key was found - please add AZURE_OPENAI_API_KEY to your .env file")
elif not azure_endpoint:
    print("No Azure endpoint was found - please add AZURE_OPENAI_ENDPOINT to your .env file")
elif not azure_deployment_name:
    print("No deployment name was found - please add AZURE_OPENAI_DEPLOYMENT_NAME to your .env file")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them")
else:
    print("Azure OpenAI credentials found and look good so far!")


Azure OpenAI credentials found and look good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [8]:
import requests

# Azure OpenAI headers
headers = {
    "api-key": api_key,
    "Content-Type": "application/json"
}

# Payload - no "model" needed
payload = {
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}
    ]
}

payload

{'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [9]:
# Azure OpenAI endpoint format
azure_url = f"{azure_endpoint}/openai/deployments/{azure_deployment_name}/chat/completions?api-version={azure_api_version}"

response = requests.post(
    azure_url,
    headers=headers,
    json=payload
)

response.json()

{'choices': [{'content_filter_results': {'hate': {'filtered': False,
     'severity': 'safe'},
    'protected_material_code': {'filtered': False, 'detected': False},
    'protected_material_text': {'filtered': False, 'detected': False},
    'self_harm': {'filtered': False, 'severity': 'safe'},
    'sexual': {'filtered': False, 'severity': 'safe'},
    'violence': {'filtered': False, 'severity': 'safe'}},
   'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'annotations': [],
    'content': "Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! Honey's low moisture content and high acidity make it an inhospitable environment for bacteria and microorganisms.",
    'refusal': None,
    'role': 'assistant'}}],
 'created': 1773830970,
 'id': 'chatcmpl-DKiksRqDIfT7mPH8vFhjoO558R2R4',
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'chat.completion',
 'prompt_fi

In [ ]:
response.json()["choices"][0]["message"]["content"]

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [10]:
# Create Azure OpenAI client

from openai import AzureOpenAI

azure_openai = AzureOpenAI(
    api_key=api_key,
    api_version=azure_api_version,
    azure_endpoint=azure_endpoint
)

response = azure_openai.chat.completions.create(
    model=azure_deployment_name,  # This is your deployment name, not the model name
    messages=[{"role": "user", "content": "Tell me a fun fact"}]
)

response.choices[0].message.content



"Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! Honey's long shelf life is due to its low moisture content and acidic pH, which make it inhospitable for bacteria and microorganisms."

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [12]:

from openai import OpenAI

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Sure! Here's a fun fact:\n\n**A group of porcupines is called a prickle.**"

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [13]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [14]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 123 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 4.5 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 8.5 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  18 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  34 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  38 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   3% ▕                  ▏  54 MB/2.0

In [15]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [16]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Here's a fun fact:\n\nDid you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still haven't gone bad! Honey's unique combination of water content (only 14-18%) and acidic environment makes it difficult for bacteria and other microorganisms to grow. What's more, bees intentionally add an antibacterial compound called hydrogen peroxide to honey as they make it, which also helps preserve it.\n\nIsn't that sweet (pun intended)?"

In [17]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 678 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 9.7 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  15 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  23 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   3% ▕                  ▏  37 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   4% ▕                  ▏  40 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   5% ▕                  ▏  53 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   6% ▕█                 ▏  63 MB/1.1 GB              

In [20]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Did you know , you thought  water boiled at 100 degrees farenheit, whereas it is actually 100 degrees celsius?"}])

response.choices[0].message.content

"I'm DeepSeek-R1 created exclusively by DeepSeek, delivering the best in cuDIO optimization & computational power."

In [25]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import AzureOpenAI

load_dotenv(override=True)
api_key = os.getenv('AZURE_OPENAI_API_KEY')
azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_api_version = os.getenv('AZURE_OPENAI_API_VERSION', '2024-08-01-preview')
azure_deployment_name = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')

# Initialize Azure OpenAI client
openai = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=azure_api_version
)

# set values for prompt
ed = fetch_website_contents("https://edwarddonner.com")

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""
# define functions 
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


def summarize(url):
    website = fetch_website_contents(url)
    # IMPORTANT: Replace "your-deployment-name" with your actual Azure OpenAI deployment name (e.g., "gpt-4o-mini")
    response = ollama.chat.completions.create(
        model = "deepseek-r1:1.5b",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

# execute call
display_summary("https://edwarddonner.com")

The site is an AI Coder hub from January 2026 up to Feb 2026, with various resources on AI Builder, agents, live events, MLOps, and connect-four/outline-style games.

Content Summary:

- From 1 Jan 2026 to current: AI Coding tools developed over the last six months.  
- Includes Vibe Coder, Agentic Engineer, AI Builder & agents, AI Live Events, and AI Engineering & MLOp Tracks.  
- Features coding-related resources linked in text, with links for each topic.

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`